# Attention Fusion Experiment: RGB + Depth + Joint Encoders

## Objective
This notebook implements a **multimodal attention-based fusion model** for posture classification using three pretrained encoders:

- **RGB encoder**
- **Depth encoder**
- **Joint encoder**

The goal is to combine feature representations from all three modalities and train a **attention0based fusion classification head** on top of them.

---

## Experiment Setup
In this experiment:

- pretrained encoder weights are loaded from saved artifacts
- encoder backbones are used as **feature extractors**
- encoder parameters are initially **frozen**
- An **attention-based fusion module**  is trained to model interactions between modality-specific features
- the final classifier predicts one of the posture classes:
  - **supine**
  - **left**
  - **right**

---

## Why Attention Fusion
Attention fusion allows the model to learn how the modalities relate to one another for each sample, rather than simply combining them with fixed importance.

This is useful because:

- **RGB** provides rich appearance cues but may degrade under blanket occlusion
- **Depth** provides spatial structure that may remain informative when RGB is weak
- **Joint features** features provide body configuration cues that can complement image-based features

The attention mechanism helps the model learn:

which modality should be emphasized
how one modality should influence the interpretation of another
how to form a stronger joint representation from all three modalities before final classification

This makes attention fusion a strong choice for multimodal posture classification, especially when modality reliability varies across conditions.

## Imports and Setup

In [1]:
# =========================
# Core
# =========================
from pathlib import Path

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models


# =========================
# Required Libraries
# =========================
import pandas as pd
import numpy as np
import json
import scipy.io as sio
import random
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

In [2]:
# =========================
# Device setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Paths and Configuration

In [3]:
# =========================
# Paths
# =========================
CWD = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in [CWD, *CWD.parents] if (path / "requirements.txt").exists()), CWD)

ARTIFACTS_DIR = PROJECT_ROOT / "experiments" / "artifacts" / "vision-spatial"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

DEPTH_ARTIFACT_DIR = ARTIFACTS_DIR / "encoders" / "depth_encoder_cnn" / "checkpoints"
RGB_ARTIFACT_DIR = ARTIFACTS_DIR / "encoders" / "rgb_encoder_cnn" / "checkpoints"
JOINT_ARTIFACT_DIR = ARTIFACTS_DIR / "encoders" / "joint_xyo" / "checkpoints"

FUSION_ARTIFACT_DIR = ARTIFACTS_DIR / "fusion" / "attention_fusion"
FUSION_CHECKPOINT_DIR = FUSION_ARTIFACT_DIR / "checkpoints"
FUSION_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = PROJECT_ROOT.parent / "SLP2022" / "SLP" / "danaLab"
CSV_PATH = DATASET_ROOT / "posture_labels_all_modalities.csv"

FUSION_RESULTS_DIR = FUSION_ARTIFACT_DIR / "results"
FUSION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Depth artifacts:", DEPTH_ARTIFACT_DIR)
print("RGB artifacts:", RGB_ARTIFACT_DIR)
print("Joint artifacts:", JOINT_ARTIFACT_DIR)
print("Fusion checkpoints:", FUSION_CHECKPOINT_DIR)

Project root: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense
Depth artifacts: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\depth_encoder_cnn\checkpoints
RGB artifacts: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints
Joint artifacts: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\joint_xyo\checkpoints
Fusion checkpoints: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\checkpoints_fusion_gated


## Experiment Configuration

Define the basic experiment settings for gated fusion, including:

- number of classes
- batch size
- learning rate
- number of epochs
- common feature dimension for fusion

These can be adjusted later if needed.

In [4]:
# =========================
# Experiment config
# =========================
NUM_CLASSES = 3

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

COMMON_FEATURE_DIM = 256 # to convert all features from encoders to a common size
DROPOUT = 0.3

CLASS_NAMES = ["supine", "left", "right"]

## Encoder Checkpoint Paths

In [5]:
# =========================
# Encoder checkpoint files
# =========================
DEPTH_CHECKPOINT_PATH = DEPTH_ARTIFACT_DIR / "best_depth_encoder_only.pth"
RGB_CHECKPOINT_PATH = RGB_ARTIFACT_DIR / "best_rgb_encoder.pt"
JOINT_CHECKPOINT_PATH = JOINT_ARTIFACT_DIR / "joint_encoder_xyo_RGB.pth"

print("Depth checkpoint:", DEPTH_CHECKPOINT_PATH)
print("RGB checkpoint:", RGB_CHECKPOINT_PATH)
print("Joint checkpoint:", JOINT_CHECKPOINT_PATH)

Depth checkpoint: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\depth_encoder_cnn\checkpoints\best_depth_encoder_only.pth
RGB checkpoint: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\rgb_encoder_cnn\checkpoints\best_rgb_encoder.pt
Joint checkpoint: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\joint_xyo\checkpoints\joint_encoder_xyo_RGB.pth


In [6]:
# =========================
# Dataset and metadata
# =========================

def build_fusion_metadata(csv_path):
    df = pd.read_csv(csv_path)
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # use RGB rows as the aligned base sample definition
    rgb_df = df[df["modality"] == "RGB"].copy()

    rgb_df["rgb_path"] = (
        rgb_df["subject_id"] + "/RGB/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )

    rgb_df["depth_path"] = (
        rgb_df["subject_id"] + "/depth/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )

    # joints shared at subject level
    rgb_df["joint_file"] = rgb_df["subject_id"] + "/joints_gt_RGB.mat"
    rgb_df["frame_idx_0based"] = rgb_df["image_index"] - 1

    return rgb_df[[
        "subject_id",
        "condition",
        "image_index",
        "label",
        "label_id",
        "rgb_path",
        "depth_path",
        "joint_file",
        "frame_idx_0based",
    ]].reset_index(drop=True)

# =========================
# Fusion dataset and data loading
# =========================

IMG_W = 576.0
IMG_H = 1024.0

def preprocess_joint_frame_xyo(frame_joints):
    x = frame_joints[0].astype(np.float32) / IMG_W
    y = frame_joints[1].astype(np.float32) / IMG_H
    occ = frame_joints[2].astype(np.float32)
    out = np.stack([x, y, occ], axis=1)
    return out.reshape(-1)

rgb_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

depth_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

class FusionDataset(Dataset):
    def __init__(self, df, dataset_root, rgb_transform=None, depth_transform=None):
        self.df = df.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.rgb_transform = rgb_transform
        self.depth_transform = depth_transform
        self.joint_cache = {}

    def __len__(self):
        return len(self.df)

    def _load_joint_file(self, joint_rel_path):
        if joint_rel_path not in self.joint_cache:
            mat = sio.loadmat(self.dataset_root / joint_rel_path)
            self.joint_cache[joint_rel_path] = mat["joints_gt"]
        return self.joint_cache[joint_rel_path]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rgb_img = Image.open(self.dataset_root / row["rgb_path"]).convert("RGB")
        if self.rgb_transform is not None:
            rgb_img = self.rgb_transform(rgb_img)

        depth_img = Image.open(self.dataset_root / row["depth_path"]).convert("L")
        if self.depth_transform is not None:
            depth_img = self.depth_transform(depth_img)

        joints_all = self._load_joint_file(row["joint_file"])
        frame = joints_all[:, :, int(row["frame_idx_0based"])]
        joint_vec = preprocess_joint_frame_xyo(frame)

        return {
            "rgb": rgb_img,
            "depth": depth_img,
            "joint": torch.tensor(joint_vec, dtype=torch.float32),
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
            "condition": row["condition"],
            "subject_id": row["subject_id"],
        }
    

# =========================
# Loader split
# =========================

def subject_wise_split(subject_ids, train_ratio=0.70, val_ratio=0.15, seed=42):
    subject_ids = sorted(subject_ids)
    rng = random.Random(seed)
    rng.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects


fusion_df = build_fusion_metadata(CSV_PATH)

subjects = sorted(fusion_df["subject_id"].unique().tolist())
train_subjects, val_subjects, test_subjects = subject_wise_split(subjects, 0.70, 0.15, 42)

train_df = fusion_df[fusion_df["subject_id"].isin(train_subjects)].reset_index(drop=True)
val_df   = fusion_df[fusion_df["subject_id"].isin(val_subjects)].reset_index(drop=True)
test_df  = fusion_df[fusion_df["subject_id"].isin(test_subjects)].reset_index(drop=True)

train_dataset = FusionDataset(train_df, DATASET_ROOT, rgb_transform, depth_transform)
val_dataset   = FusionDataset(val_df, DATASET_ROOT, rgb_transform, depth_transform)
test_dataset  = FusionDataset(test_df, DATASET_ROOT, rgb_transform, depth_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Encoder Model Definitions

Define the encoder architectures needed to load the pretrained weights.

At this stage, we only recreate the model structures required for:

- RGB encoder
- Depth encoder
- Joint encoder

These definitions should match the architectures used during the original encoder training notebooks.

In [7]:
# =========================
# Encoder feature dimensions
# =========================
DEPTH_FEATURE_DIM = 512
RGB_FEATURE_DIM = 128
JOINT_INPUT_DIM = 42
JOINT_FEATURE_DIM = 128

In [8]:
class DepthEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)

        # change first conv to 1 channel
        backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        # remove FC
        layers = list(backbone.children())[:-1]

        self.encoder = nn.Sequential(*layers)
        self.flatten = nn.Flatten()

        self.feature_dim = 512

    def forward(self, x):
        x = self.encoder(x)
        x = self.flatten(x)
        return x


class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128)
        )
        self.feature_dim = 128

    def forward(self, x):
        x = self.backbone(x)
        x = self.projection(x)
        return x


class JointEncoder(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, feature_dim=128, dropout=0.3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.feature_dim = feature_dim

    def forward(self, x):
        return self.encoder(x)

## Load Pretrained Encoders

Instantiate the three encoder models and load their encoder-only weights from the saved checkpoints.

After loading:
- move models to the selected device
- set them to evaluation mode
- freeze parameters so that only the gated fusion head is trained in this first experiment stage

In [9]:
# =========================
# Instantiate encoders
# =========================
depth_encoder = DepthEncoder().to(device)
rgb_encoder = RGBEncoder().to(device)
joint_encoder = JointEncoder(
    input_dim=JOINT_INPUT_DIM,
    hidden_dim=128,
    feature_dim=JOINT_FEATURE_DIM,
    dropout=0.3
).to(device)

# =========================
# Load checkpoints
# =========================
depth_ckpt = torch.load(DEPTH_CHECKPOINT_PATH, map_location=device)
rgb_ckpt = torch.load(RGB_CHECKPOINT_PATH, map_location=device)
joint_ckpt = torch.load(JOINT_CHECKPOINT_PATH, map_location=device)

depth_encoder.encoder.load_state_dict(depth_ckpt["encoder_state_dict"])
rgb_encoder.load_state_dict(rgb_ckpt["encoder_state_dict"])
joint_encoder.encoder.load_state_dict(joint_ckpt["encoder_state_dict"])

# =========================
# Freeze encoders
# =========================
for model in [depth_encoder, rgb_encoder, joint_encoder]:
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("All encoders loaded and frozen successfully.")

All encoders loaded and frozen successfully.


# Attention Fusion Model

Report Helper Functions

In [10]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def save_history_csv(history_rows, path):
    pd.DataFrame(history_rows).to_csv(path, index=False)

def save_confusion_matrix_csv(cm, path):
    pd.DataFrame(cm).to_csv(path, index=False)

def save_confusion_matrix_figure(cm, class_names, path):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Joint + RGB + Depth Attention Fusion Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")

    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def save_split_subjects(train_subjects, val_subjects, test_subjects, path):
    split_info = {
        "train_subjects": train_subjects,
        "val_subjects": val_subjects,
        "test_subjects": test_subjects,
        "num_train_subjects": len(train_subjects),
        "num_val_subjects": len(val_subjects),
        "num_test_subjects": len(test_subjects),
    }
    save_json(split_info, path)

In [11]:
# =========================
# Attention-based fusion head
# =========================
class AttentionFusionClassifier(nn.Module):
    def __init__(
        self,
        depth_encoder,
        rgb_encoder,
        joint_encoder,
        depth_feature_dim=512,
        rgb_feature_dim=128,
        joint_feature_dim=128,
        common_feature_dim=256,
        num_heads=4,
        ff_dim=512,
        dropout=0.1,
        num_classes=3,
    ):
        super().__init__()

        # frozen encoders
        self.depth_encoder = depth_encoder
        self.rgb_encoder = rgb_encoder
        self.joint_encoder = joint_encoder

        # project all modalities to same dim
        self.depth_proj = nn.Linear(depth_feature_dim, common_feature_dim)
        self.rgb_proj = nn.Linear(rgb_feature_dim, common_feature_dim)
        self.joint_proj = nn.Linear(joint_feature_dim, common_feature_dim)

        # modality-type embeddings (helps attention distinguish tokens)
        self.modality_embed = nn.Parameter(torch.randn(3, common_feature_dim))

        # one small transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=common_feature_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.attention_block = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(common_feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, depth_x, rgb_x, joint_x, return_features=False):
        with torch.no_grad():
            f_depth = self.depth_encoder(depth_x)   # [B, 512]
            f_rgb = self.rgb_encoder(rgb_x)         # [B, 128]
            f_joint = self.joint_encoder(joint_x)   # [B, 128]

        t_depth = self.depth_proj(f_depth)
        t_rgb = self.rgb_proj(f_rgb)
        t_joint = self.joint_proj(f_joint)

        # stack as 3 modality tokens
        tokens = torch.stack([t_rgb, t_depth, t_joint], dim=1)   # [B, 3, D]

        # add modality embeddings
        tokens = tokens + self.modality_embed.unsqueeze(0)

        # self-attention across modalities
        tokens = self.attention_block(tokens)  # [B, 3, D]

        # pool tokens
        fused = tokens.mean(dim=1)  # [B, D]

        logits = self.classifier(fused)

        if return_features:
            return logits, fused
        return logits
    
# =========================
# Creating fusion model
# =========================
fusion_model = AttentionFusionClassifier(
    depth_encoder=depth_encoder,
    rgb_encoder=rgb_encoder,
    joint_encoder=joint_encoder,
    depth_feature_dim=DEPTH_FEATURE_DIM,
    rgb_feature_dim=RGB_FEATURE_DIM,
    joint_feature_dim=JOINT_FEATURE_DIM,
    common_feature_dim=COMMON_FEATURE_DIM,
    num_heads=4,
    ff_dim=512,
    dropout=0.1,
    num_classes=NUM_CLASSES,
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, fusion_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

trainable_params = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in fusion_model.parameters() if not p.requires_grad)

print("Trainable params:", trainable_params)
print("Frozen params   :", frozen_params)

Trainable params: 758531
Frozen params   : 22532992


C:\Users\John joseph peter\AppData\Local\Temp\ipykernel_28928\2255670998.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.attention_block = nn.TransformerEncoder(encoder_layer, num_layers=1)


In [13]:
def train_one_epoch_fusion(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        depth_x = batch["depth"].to(device)
        rgb_x = batch["rgb"].to(device)
        joint_x = batch["joint"].to(device)
        y = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(depth_x, rgb_x, joint_x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)

        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")

    return avg_loss, acc, f1

# =========================
# Evaluation Block
# =========================
@torch.no_grad()
def evaluate_fusion(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        depth_x = batch["depth"].to(device)
        rgb_x = batch["rgb"].to(device)
        joint_x = batch["joint"].to(device)
        y = batch["label"].to(device)

        logits = model(depth_x, rgb_x, joint_x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)

        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    
    return avg_loss, acc, f1, cm, report


# =========================
# Main Training block
# =========================

best_val_f1 = -1.0
best_epoch = -1
best_ckpt_path = FUSION_CHECKPOINT_DIR / "best_attention_fusion.pth"

history_rows = []
# save split once
save_split_subjects(
    train_subjects,
    val_subjects,
    test_subjects,
    FUSION_RESULTS_DIR / "attention_fusion_split_subjects.json"
)


for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch_fusion(
        fusion_model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc, val_f1, val_cm, val_report = evaluate_fusion(
        fusion_model, val_loader, criterion, device
    )

    history_rows.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"[Train] loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f}")
    print(f"[Val]   loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": fusion_model.state_dict(),
                "config": {
                    "common_feature_dim": COMMON_FEATURE_DIM,
                    "num_classes": NUM_CLASSES,
                    "num_heads": 4,
                    "ff_dim": 512,
                    "dropout": 0.1,
                },
                "best_val_f1": best_val_f1,
                "best_epoch": best_epoch,
            },
            best_ckpt_path,
        )

        print(f"Saved best model -> {best_ckpt_path}")

# save training history
save_history_csv(history_rows, FUSION_RESULTS_DIR / "attention_fusion_history.csv")

# =========================
# Final evaluation on test set
# =========================
ckpt = torch.load(best_ckpt_path, map_location=device)
fusion_model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_acc, test_f1, test_cm, test_report = evaluate_fusion(
    fusion_model, test_loader, criterion, device
)

print("\n[Test]")
print(f"loss={test_loss:.4f} acc={test_acc:.4f} f1={test_f1:.4f}")
print("Confusion matrix:")
print(test_cm)

# =========================
# Save test results
# =========================

# save confusion matrix
save_confusion_matrix_csv(
    test_cm,
    FUSION_RESULTS_DIR / "attention_fusion_confusion_matrix.csv"
)
save_confusion_matrix_figure(
    test_cm,
    CLASS_NAMES,
    FUSION_RESULTS_DIR / "attention_fusion_confusion_matrix.png"
)

# save final metrics json
final_metrics = {
    "experiment_name": "attention_fusion_fusion",
    "best_epoch": best_epoch,
    "best_val_macro_f1": best_val_f1,
    "test_loss": test_loss,
    "test_accuracy": test_acc,
    "test_macro_f1": test_f1,
    "classwise_metrics": test_report,
    "checkpoint_path": str(best_ckpt_path),
    "config": {
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "common_feature_dim": COMMON_FEATURE_DIM,
        "dropout": DROPOUT,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "depth_feature_dim": DEPTH_FEATURE_DIM,
        "rgb_feature_dim": RGB_FEATURE_DIM,
        "joint_feature_dim": JOINT_FEATURE_DIM,
        "joint_input_dim": JOINT_INPUT_DIM,
    },
    "num_train_samples": len(train_df),
    "num_val_samples": len(val_df),
    "num_test_samples": len(test_df),
}

save_json(
    final_metrics,
    FUSION_RESULTS_DIR / "attention_fusion_test_metrics.json"
)

# save compact summary json
experiment_summary = {
    "model_name": "AttentionFusionClassifier",
    "modalities": ["RGB", "Depth", "Joint"],
    "encoders_frozen": True,
    "fusion_type": "self_attention",
    "num_heads": 4,
    "common_feature_dim": COMMON_FEATURE_DIM,
    "num_classes": NUM_CLASSES,
    "labels": {
        "0": "supine",
        "1": "left",
        "2": "right",
    },
    "notes": "Frozen RGB, Depth, and Joint encoders with attention-based fusion head.",
}
save_json(
    experiment_summary,
    FUSION_RESULTS_DIR / "attention_fusion_experiment_summary.json"
)

print("\nSaved report files in:", FUSION_RESULTS_DIR)


Epoch 1/15
[Train] loss=0.0139 acc=0.9971 f1=0.9971
[Val]   loss=0.0000 acc=1.0000 f1=1.0000
Saved best model -> C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\checkpoints_fusion_gated\best_attention_fusion.pth

Epoch 2/15
[Train] loss=0.0170 acc=0.9966 f1=0.9966
[Val]   loss=0.0018 acc=0.9995 f1=0.9995

Epoch 3/15
[Train] loss=0.0129 acc=0.9966 f1=0.9966
[Val]   loss=0.0003 acc=1.0000 f1=1.0000

Epoch 4/15
[Train] loss=0.0116 acc=0.9972 f1=0.9972
[Val]   loss=0.0004 acc=1.0000 f1=1.0000

Epoch 5/15
[Train] loss=0.0117 acc=0.9966 f1=0.9966
[Val]   loss=0.0005 acc=1.0000 f1=1.0000

Epoch 6/15
[Train] loss=0.0114 acc=0.9977 f1=0.9977
[Val]   loss=0.0004 acc=1.0000 f1=1.0000

Epoch 7/15
[Train] loss=0.0112 acc=0.9974 f1=0.9974
[Val]   loss=0.0002 acc=1.0000 f1=1.0000

Epoch 8/15
[Train] loss=0.0104 acc=0.9977 f1=0.9977
[Val]   loss=0.0001 acc=1.0000 f1=1.0000

Epoch 9/15
[Train] loss=0.0136 acc=0.9966 f1=0.9966
[Val]   loss=0.0003 acc=1.0000 f1=1.0000

Epoch 10/15

## Final Pipeline with attention-fusion

In [14]:
# =========================
# Selective unfreezing helpers
# =========================
def unfreeze_module(module):
    for p in module.parameters():
        p.requires_grad = True

def freeze_module(module):
    for p in module.parameters():
        p.requires_grad = False

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [15]:
# =========================
# Start from frozen best attention model
# =========================
finetune_ckpt_path = FUSION_CHECKPOINT_DIR / "best_attention_fusion_finetuned.pth"

# load best frozen attention checkpoint
ckpt = torch.load(best_ckpt_path, map_location=device)
fusion_model.load_state_dict(ckpt["model_state_dict"])

# by default freeze everything first
freeze_module(fusion_model.depth_encoder)
freeze_module(fusion_model.rgb_encoder)
freeze_module(fusion_model.joint_encoder)

# unfreeze fusion head parts
unfreeze_module(fusion_model.depth_proj)
unfreeze_module(fusion_model.rgb_proj)
unfreeze_module(fusion_model.joint_proj)
unfreeze_module(fusion_model.attention_block)
unfreeze_module(fusion_model.classifier)

# unfreeze top RGB block
unfreeze_module(fusion_model.rgb_encoder.backbone.layer4)

# unfreeze last part of depth encoder
# depth_encoder.encoder is Sequential(*layers[:-1]) from ResNet18 children
# unfreeze the last 2 modules of that Sequential as a practical approximation
for m in list(fusion_model.depth_encoder.encoder.children())[-2:]:
    unfreeze_module(m)

# keep joint encoder frozen for first finetune run
freeze_module(fusion_model.joint_encoder)

print("Trainable params after partial unfreeze:", count_trainable_params(fusion_model))

Trainable params after partial unfreeze: 17545987


In [16]:
# =========================
# Differential learning rates
# =========================
FUSION_HEAD_LR = 1e-3
ENCODER_LR = 1e-4
WEIGHT_DECAY = 1e-4
FINETUNE_EPOCHS = 10

optimizer_finetune = torch.optim.Adam([
    # fusion head / projections
    {"params": fusion_model.depth_proj.parameters(), "lr": FUSION_HEAD_LR},
    {"params": fusion_model.rgb_proj.parameters(), "lr": FUSION_HEAD_LR},
    {"params": fusion_model.joint_proj.parameters(), "lr": FUSION_HEAD_LR},
    {"params": fusion_model.attention_block.parameters(), "lr": FUSION_HEAD_LR},
    {"params": fusion_model.classifier.parameters(), "lr": FUSION_HEAD_LR},

    # partially unfrozen encoders
    {"params": fusion_model.rgb_encoder.backbone.layer4.parameters(), "lr": ENCODER_LR},
    {"params": [p for m in list(fusion_model.depth_encoder.encoder.children())[-2:] for p in m.parameters()], "lr": ENCODER_LR},
], weight_decay=WEIGHT_DECAY)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [17]:
# =========================
# Fine-tuning stage
# =========================
best_val_f1_ft = -1.0
best_epoch_ft = -1
finetune_history = []

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch_fusion(
        fusion_model, train_loader, optimizer_finetune, criterion, device
    )

    val_loss, val_acc, val_f1, val_cm, val_report = evaluate_fusion(
        fusion_model, val_loader, criterion, device
    )

    finetune_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    print(f"\n[Fine-tune] Epoch {epoch+1}/{FINETUNE_EPOCHS}")
    print(f"[Train] loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f}")
    print(f"[Val]   loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}")

    if val_f1 > best_val_f1_ft:
        best_val_f1_ft = val_f1
        best_epoch_ft = epoch + 1

        torch.save(
            {
                "model_state_dict": fusion_model.state_dict(),
                "best_val_f1": best_val_f1_ft,
                "best_epoch": best_epoch_ft,
                "stage": "finetune",
                "config": {
                    "fusion_head_lr": FUSION_HEAD_LR,
                    "encoder_lr": ENCODER_LR,
                    "finetune_epochs": FINETUNE_EPOCHS,
                    "common_feature_dim": COMMON_FEATURE_DIM,
                    "num_classes": NUM_CLASSES,
                    "num_heads": 4,
                    "ff_dim": 512,
                    "dropout": 0.1,
                    "joint_encoder_frozen": True,
                },
            },
            finetune_ckpt_path,
        )

        print(f"Saved best fine-tuned model -> {finetune_ckpt_path}")


[Fine-tune] Epoch 1/10
[Train] loss=0.0155 acc=0.9965 f1=0.9965
[Val]   loss=0.0008 acc=1.0000 f1=1.0000
Saved best fine-tuned model -> C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\checkpoints_fusion_gated\best_attention_fusion_finetuned.pth

[Fine-tune] Epoch 2/10
[Train] loss=0.0140 acc=0.9961 f1=0.9961
[Val]   loss=0.0005 acc=1.0000 f1=1.0000

[Fine-tune] Epoch 3/10
[Train] loss=0.0139 acc=0.9966 f1=0.9966
[Val]   loss=0.0009 acc=1.0000 f1=1.0000

[Fine-tune] Epoch 4/10
[Train] loss=0.0122 acc=0.9976 f1=0.9976
[Val]   loss=0.0061 acc=0.9985 f1=0.9985

[Fine-tune] Epoch 5/10
[Train] loss=0.0117 acc=0.9974 f1=0.9974
[Val]   loss=0.0053 acc=0.9965 f1=0.9965

[Fine-tune] Epoch 6/10
[Train] loss=0.0105 acc=0.9971 f1=0.9971
[Val]   loss=0.0001 acc=1.0000 f1=1.0000

[Fine-tune] Epoch 7/10
[Train] loss=0.0110 acc=0.9977 f1=0.9977
[Val]   loss=0.0002 acc=1.0000 f1=1.0000

[Fine-tune] Epoch 8/10
[Train] loss=0.0116 acc=0.9972 f1=0.9972
[Val]   loss=0.0022 acc=1.0000

In [18]:
# =========================
# Final evaluation of fine-tuned model
# =========================
ckpt_ft = torch.load(finetune_ckpt_path, map_location=device)
fusion_model.load_state_dict(ckpt_ft["model_state_dict"])

test_loss_ft, test_acc_ft, test_f1_ft, test_cm_ft, test_report_ft = evaluate_fusion(
    fusion_model, test_loader, criterion, device
)

print("\n[Fine-tuned Test]")
print(f"loss={test_loss_ft:.4f} acc={test_acc_ft:.4f} f1={test_f1_ft:.4f}")
print("Confusion matrix:")
print(test_cm_ft)


[Fine-tuned Test]
loss=0.0103 acc=0.9963 f1=0.9963
Confusion matrix:
[[720   0   0]
 [  4 716   0]
 [  4   0 716]]


In [19]:
# =========================
# Save fine-tuning report artifacts
# =========================
FUSION_FINETUNE_RESULTS_DIR = ARTIFACTS_DIR / "results_attention_fusion_finetuned"
FUSION_FINETUNE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# history csv
save_history_csv(
    finetune_history,
    FUSION_FINETUNE_RESULTS_DIR / "joint_attention_finetuned_history.csv"
)

# confusion matrix
save_confusion_matrix_csv(
    test_cm_ft,
    FUSION_FINETUNE_RESULTS_DIR / "joint_attention_finetuned_confusion_matrix.csv"
)
save_confusion_matrix_figure(
    test_cm_ft,
    CLASS_NAMES,
    FUSION_FINETUNE_RESULTS_DIR / "joint_attention_finetuned_confusion_matrix.png"
)

# final metrics
finetune_metrics = {
    "experiment_name": "joint_attention_fusion_finetuned",
    "best_epoch": best_epoch_ft,
    "best_val_macro_f1": best_val_f1_ft,
    "test_loss": test_loss_ft,
    "test_accuracy": test_acc_ft,
    "test_macro_f1": test_f1_ft,
    "classwise_metrics": test_report_ft,
    "checkpoint_path": str(finetune_ckpt_path),
    "config": {
        "fusion_head_lr": FUSION_HEAD_LR,
        "encoder_lr": ENCODER_LR,
        "finetune_epochs": FINETUNE_EPOCHS,
        "common_feature_dim": COMMON_FEATURE_DIM,
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "depth_feature_dim": DEPTH_FEATURE_DIM,
        "rgb_feature_dim": RGB_FEATURE_DIM,
        "joint_feature_dim": JOINT_FEATURE_DIM,
        "joint_input_dim": JOINT_INPUT_DIM,
        "joint_encoder_frozen": True,
    },
}
save_json(
    finetune_metrics,
    FUSION_FINETUNE_RESULTS_DIR / "joint_attention_finetuned_test_metrics.json"
)

finetune_summary = {
    "model_name": "AttentionFusionClassifier",
    "stage": "end_to_end_finetune",
    "modalities": ["RGB", "Depth", "Joint"],
    "fusion_type": "self_attention",
    "encoder_unfreeze_policy": {
        "rgb": "layer4 unfrozen",
        "depth": "last 2 encoder modules unfrozen",
        "joint": "kept frozen"
    },
    "notes": "Stage-2 fine-tuning after selecting attention fusion as the best frozen-encoder benchmark head."
}
save_json(
    finetune_summary,
    FUSION_FINETUNE_RESULTS_DIR / "joint_attention_finetuned_experiment_summary.json"
)

print("\nSaved fine-tuning report files in:", FUSION_FINETUNE_RESULTS_DIR)


Saved fine-tuning report files in: C:\Users\John joseph peter\Desktop\NUS\ApneaSense\ApneaSense\artifacts\results_attention_fusion_finetuned


## Why We Retain Frozen Encoders for the Final Posture Model

Although a Stage-2 fine-tuning experiment was performed after selecting attention fusion as the best benchmark head, the fine-tuned model did not improve over the frozen-encoder version. The frozen attention-fusion model achieved slightly better test performance, while the partially fine-tuned version introduced a small degradation.

This suggests that the pretrained RGB, Depth, and Joint encoders were already providing highly discriminative and well-aligned features for the posture classification task. In this setting, additional encoder adaptation was not beneficial and likely introduced unnecessary feature drift rather than useful task-specific refinement.

For this reason, the final posture-classification pipeline uses:

attention-based fusion
frozen pretrained encoders
trainable fusion head only

This choice is preferred because it offers:

the best observed test performance
a simpler and more stable training pipeline
lower risk of overfitting
easier reproducibility and fairer benchmarking across fusion heads

Therefore, the frozen-encoder attention fusion model is retained as the final posture branch for the overall multimodal system.